In [2]:
import pandas as pd
import plotly.graph_objects as go

# 1. Definición de las funciones de simulación
def simulate_sjf(processes):
    time = 0
    completed = []
    ready_queue = []
    pending = sorted([p.copy() for p in processes], key=lambda x: x['arrival'])

    while pending or ready_queue:
        while pending and pending[0]['arrival'] <= time:
            ready_queue.append(pending.pop(0))

        if ready_queue:
            ready_queue.sort(key=lambda x: x['burst'])
            p = ready_queue.pop(0)
            p['start'] = time
            time += p['burst']
            p['finish'] = time
            p['return'] = p['finish'] - p['arrival']
            p['wait'] = p['return'] - p['burst']
            completed.append(p)
        else:
            time = pending[0]['arrival']
    return completed

def simulate_rr(processes, quantum):
    time = 0
    completed = []
    queue = []
    pending = sorted([p.copy() for p in processes], key=lambda x: x['arrival'])
    burst_remaining = {p['id']: p['burst'] for p in processes}

    while pending or queue:
        while pending and pending[0]['arrival'] <= time:
            queue.append(pending.pop(0))

        if queue:
            p = queue.pop(0)
            execute = min(burst_remaining[p['id']], quantum)
            time += execute
            burst_remaining[p['id']] -= execute

            while pending and pending[0]['arrival'] <= time:
                queue.append(pending.pop(0))

            if burst_remaining[p['id']] > 0:
                queue.append(p)
            else:
                p['finish'] = time
                p['return'] = p['finish'] - p['arrival']
                p['wait'] = p['return'] - p['burst']
                completed.append(p)
        else:
            time = pending[0]['arrival']
    return completed

# 2. Función para graficar tablas elegantes con Plotly
def display_styled_table(df, title):
    fig = go.Figure(data=[go.Table(
        header=dict(values=["<b>Proceso</b>", "<b>T. Llegada</b>", "<b>Ráfaga</b>",
                           "<b>Finalización</b>", "<b>T. Retorno</b>", "<b>T. Espera</b>"],
                    fill_color='royalblue',
                    font=dict(color='white', size=12),
                    align='center'),
        cells=dict(values=[df.id, df.arrival, df.burst, df.finish, df.return_t, df.wait],
                   fill_color='lavender',
                   align='center'))
    ])
    fig.update_layout(title=title, margin=dict(l=20, r=20, t=60, b=20))
    fig.show()

# 3. Datos del taller
data = [
    {'id': 'P1', 'arrival': 0, 'burst': 7},
    {'id': 'P2', 'arrival': 2, 'burst': 5},
    {'id': 'P3', 'arrival': 4, 'burst': 2},
    {'id': 'P4', 'arrival': 5, 'burst': 4},
    {'id': 'P5', 'arrival': 8, 'burst': 8}
]

# 4. Ejecución y presentación de resultados
# SJF
res_sjf = simulate_sjf(data)
df_sjf = pd.DataFrame(res_sjf).rename(columns={'return': 'return_t'}).sort_values('id')
display_styled_table(df_sjf, f"Simulación SJF - Promedio Espera: {df_sjf.wait.mean():.2f}")

# Round Robin
res_rr = simulate_rr(data, 3)
df_rr = pd.DataFrame(res_rr).rename(columns={'return': 'return_t'}).sort_values('id')
display_styled_table(df_rr, f"Simulación Round Robin (Q=3) - Promedio Espera: {df_rr.wait.mean():.2f}")